# 業績情報取得

`stock/irbank.csv`（企業ID・URL取得済みの内国株式銘柄）を対象に、IRBANK（irbank.net）の決算ページ（`/results`）から、高配当ラベル判定に使う業績指標の年別推移を取得し、`stock/financials.csv`を更新する。

## 取得する指標（主要8項目）

旧実装（`#Python/250811_株式投資自動化/Python/01_Code/plan01_基本設計.ipynb`）で定義した「主要8項目」のうち、1株配当以外の7項目＋ROEを取得する。

| metric | 指標 | 単位（`value`列） | 旧実装の選定基準 |
|---|---|---|---|
| `revenue` | 売上高（銀行は経常収益、損保は正味収入保険料、IFRSは売上収益／収益） | 円 | 過去10年右肩上がりか、乱高下が激しくないか |
| `eps` | EPS（1株当たり利益） | 円 | 右肩上がりか（最重要） |
| `op_margin` | 営業利益率 | % | 平均7%、10%なら良好、5%以下はアウト |
| `roe` | ROE | % | 営業利益率が無い銘柄（銀行・保険等）の代替指標 |
| `equity_ratio` | 自己資本比率（IFRSは株主資本比率） | % | 40%以上、50%前後なら◎、80%以上ならかなり良い |
| `op_cf` | 営業活動によるCF | 円 | 毎年黒字かつ増加傾向。赤字の年があればNG |
| `cash` | 現金等 | 円 | 増加傾向か |
| `payout` | 配当性向 | % | 30〜50%健全、70〜80%厳しい、100%超えは無理している |

8項目目の**1株配当**（減配・無配の有無、成長性）は、既に`notebooks/C04_配当情報取得.ipynb`が`stock/dividends.csv`に取得しているため、ここでは取得しない（同じデータを2か所に持つと、8766の株式分割対応のような手動補正が片方にしか反映されない不整合が起きるため）。減配判定には`stock/dividends.csv`を使う。

## 旧Notebookからの変更点

- 入力を`../00_data/01_IDmap.csv`から`stock/irbank.csv`（`notebooks/C01_IRBANK企業ID取得.ipynb`が生成）に変更した（C04と同じ）。
- 保存形式を「年×銘柄コードの横持ち表を指標ごとに9ファイル」から、1ファイル・1行=1レコードの縦持ち形式（`stock/financials.csv`。列：`code, metric, year, value, is_forecast, updated_at`）に変更した。
- 抽出ロジックを、見出し直後のテキストを正規表現で区切る方式から、C04と同じ`dl.gdl`（dt=決算期、dd=値）を直接読む方式に変更した。あわせて旧実装の不具合を修正した：
  - 金額の「兆」単位に未対応だった（例：NTTの売上高「13兆7047億」が「7047億」として読まれていた）。
  - 2017年以降・直近10年分だけに切り詰めていたのをやめ、全期間を保持する（C04と同じ方針）。
  - 予想値（「予」）を実績と区別せず混ぜていたのを、`is_forecast`列で区別するようにした（同じ年は実績優先、C04と同じ規則）。

## ローカル（Jupyter）実行専用の理由

C01・C04と同じ（IRBANKがGitHub ActionsのIPアドレスを一律ブロックしているため）。

## 既知の制約

1ページから全指標を読むため、1銘柄につき1回のアクセスで済む。ただし、どの指標も取得できなかった銘柄（ページ構成が想定外等）は`stock/financials.csv`に何も書き込まれず、実行のたびに再アクセスの対象になる（C04と同じ制約）。

## 実行後にすること

更新された`app_data/stock/financials.csv`を、データリポジトリ（`palmelo2nd/app_data`）に自分でコミット・pushする（このnotebookは保存のみ行い、git操作はしない）。

In [ ]:
import random
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests

## 設定

In [ ]:
# データリポジトリのローカルパス（notebooks/ から見た相対パス。app/stock/notebooks/ -> app_data/stock/）
DATA_DIR = Path("../../../app_data/stock")
IRBANK_CSV = DATA_DIR / "irbank.csv"          # 入力：企業ID・URL（notebooks/C01_IRBANK企業ID取得.ipynbが生成）
LABELS_CSV = DATA_DIR / "labels.csv"          # 入力：HIGH_DIV_ONLY=True のときの対象絞り込み
FINANCIALS_CSV = DATA_DIR / "financials.csv"  # 出力：年別業績指標（縦持ち）

USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
ACCEPT_HEADER = "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"

SLEEP_SEC = 1.2  # 銘柄ごとの取得間隔（秒）。IRBANK側への負荷配慮
SAVE_EVERY = 50  # この件数処理するごとに逐次保存する

JST = timezone(timedelta(hours=9))
FINANCIAL_COLUMNS = ["code", "metric", "year", "value", "is_forecast", "updated_at"]

# True にすると、既に取得済み（financials.csvに行がある）銘柄も含めて全件を再取得する
# （決算期をまたいだ後に、予想値→実績値の更新や新しい年度を取り込み直す用途。年1回程度を想定）
FORCE_REFRESH_ALL = False

# True にすると、labels.csv で L_高配当=1 の銘柄だけを対象にする（全件は約3,700銘柄×1.2秒≒75分以上かかるため、まず高配当候補だけ取りたい場合用）
HIGH_DIV_ONLY = False

# 証券コードを直接指定したい場合（例：["9432", "8306"]）。指定時は FORCE_REFRESH_ALL・HIGH_DIV_ONLY より優先し、取得済みでも再取得する
TARGET_CODES = None

# テスト用：Noneなら全件、数値を入れるとその件数だけ処理して様子を見られる
TEST_LIMIT = None

## 関数定義

指標ごとの見出し（h2）の文言で`dl.gdl`を特定する（div idは銘柄によってずれるため使わない。C04の`find_dividend_dl`と同じ考え方）。見出しの表記は業種・会計基準で異なるため、`METRIC_HEADINGS`に候補を優先順で並べている（売上高系は、どの候補にも一致しなければ業績セクション先頭のグラフを採用）。

In [ ]:
def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT, "Accept": ACCEPT_HEADER})
    return session


def fetch_html(url: str, session: requests.Session, timeout: float = 20.0) -> str:
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or "utf-8"
    return response.text


def polite_sleep(base: float) -> None:
    """次のリクエストまでジッター付きで待機する（IRBANK側への負荷配慮）。"""
    time.sleep(max(0.0, base + random.uniform(-0.5 * base, 0.5 * base)))

In [ ]:
import re

from bs4 import BeautifulSoup

# 指標キー -> h2見出しの候補（脚注記号を除いた見出しが、いずれかで始まれば一致。先頭の候補ほど優先）
METRIC_HEADINGS = {
    "revenue": ("売上高", "売上収益", "営業収益", "経常収益", "正味収入保険料", "収益"),
    "eps": ("EPS",),
    "op_margin": ("営業利益率",),
    "roe": ("ROE",),
    "equity_ratio": ("自己資本比率", "株主資本比率", "親会社所有者帰属持分比率"),  # 後2つはIFRS採用企業の表記
    "op_cf": ("営業活動によるCF",),
    "cash": ("現金等",),
    "payout": ("配当性向",),
}
# 値の単位（円／%）。保存時の値はこの単位の数値（金額は円単位に換算済み）
METRIC_UNITS = {
    "revenue": "yen", "eps": "yen", "op_margin": "pct", "roe": "pct",
    "equity_ratio": "pct", "op_cf": "yen", "cash": "yen", "payout": "pct",
}

_UNIT_MULT = (("兆", 10**12), ("億", 10**8), ("万", 10**4))


def _normalize_space(s: str) -> str:
    s = s.replace(" ", " ")
    s = re.sub(r"[ \t\r\n]+", " ", s)
    return s.strip()


def _clean_heading(text: str) -> str:
    """「自己資本比率 #12 #13」「設備投資 #6 *」のような脚注記号を除いた見出し文字列。"""
    t = re.sub(r"#\d+", "", text)
    t = t.replace("*", "")
    return _normalize_space(t)


def parse_amount_jp(text: str) -> float | None:
    """「13兆7047億」「-1兆234億」「3757億9400万」「1234万」などを円単位の数値にする。"""
    s = _normalize_space(text).replace(",", "").replace("*", "").replace(" ", "")
    m_sign = re.match(r"^[-−▲△]", s)
    sign = -1 if m_sign else 1
    total = 0.0
    found = False
    for unit, mult in _UNIT_MULT:
        m = re.search(r"(\d+(?:\.\d+)?)" + unit, s)
        if m:
            total += float(m.group(1)) * mult
            found = True
    if not found:
        m = re.search(r"(\d+(?:\.\d+)?)", s)
        if not m:
            return None
        total = float(m.group(1))
    return sign * total


def parse_yen(text: str) -> float | None:
    """「12.61円」「* 5.4円」「-3.2円」を数値にする（円が無ければNone＝別項目の誤検知防止）。"""
    s = _normalize_space(text).replace(",", "").replace("*", "").replace(" ", "")
    m = re.search(r"([-−▲△]?\d+(?:\.\d+)?)円", s)
    if not m:
        return None
    return float(re.sub(r"^[−▲△]", "-", m.group(1)))


def parse_percent(text: str) -> float | None:
    """「43.5%」「-2.1%」を数値にする（%が無ければNone）。"""
    s = _normalize_space(text).replace(",", "").replace("％", "%").replace(" ", "")
    m = re.search(r"([-−▲△]?\d+(?:\.\d+)?)%", s)
    if not m:
        return None
    return float(re.sub(r"^[−▲△]", "-", m.group(1)))


def _parse_value(metric: str, dd_text: str) -> float | None:
    unit = METRIC_UNITS[metric]
    if unit == "pct":
        return parse_percent(dd_text)
    if metric == "eps":
        return parse_yen(dd_text)
    return parse_amount_jp(dd_text)


def _parse_year_month(dt_text: str) -> tuple[int, int] | None:
    m = re.search(r"(\d{4})/(\d{2})", dt_text)
    return (int(m.group(1)), int(m.group(2))) if m else None


def find_metric_dls(soup: BeautifulSoup) -> dict:
    """指標キー -> dl.gdl 要素。見出し（h2）の文言で特定する（div idは銘柄によってずれるため使わない）。"""
    h2s = []
    for h2 in soup.find_all("h2"):
        container = h2.find_parent("div")
        dl = container.find("dl", class_="gdl") if container else None
        if dl is not None:
            h2s.append((_clean_heading(h2.get_text(" ", strip=True)), dl))

    found = {}
    for metric, keywords in METRIC_HEADINGS.items():
        for kw in keywords:
            hit = next((dl for heading, dl in h2s if heading.startswith(kw)), None)
            if hit is not None:
                found[metric] = hit
                break
    # 売上高系の見出しが想定外の名称でも、業績セクション先頭（=売上高相当）のグラフを採用する
    if "revenue" not in found and h2s:
        found["revenue"] = h2s[0][1]
    return found


def parse_metric_dl(metric: str, dl) -> dict[int, tuple[float, bool]]:
    """dl.gdl（dt=決算期、dd=値）から 年 -> (値, 予想かどうか) を返す。年ごとに実績優先・同条件なら最新月優先（C04と同じ規則）。"""
    best: dict[int, tuple[int, bool, float]] = {}
    current_dt: str | None = None
    for child in dl.find_all(["dt", "dd"], recursive=False):
        if child.name == "dt":
            current_dt = child.get_text(" ", strip=True)
            continue
        if not current_dt:
            continue
        ym = _parse_year_month(current_dt)
        is_forecast = "予" in current_dt
        current_dt = None
        if ym is None:
            continue
        value = _parse_value(metric, child.get_text(" ", strip=True))
        if value is None:  # 「-」（データ無し）など
            continue
        year, month = ym
        cur = best.get(year)
        if cur is None:
            best[year] = (month, is_forecast, value)
            continue
        cur_month, cur_forecast, _ = cur
        if cur_forecast and not is_forecast:
            best[year] = (month, is_forecast, value)
        elif (not cur_forecast) and is_forecast:
            continue
        elif month >= cur_month:
            best[year] = (month, is_forecast, value)
    return {year: (value, is_forecast) for year, (_, is_forecast, value) in best.items()}


def parse_financials(html: str) -> dict[str, dict[int, tuple[float, bool]]]:
    """IRBANK決算ページ（/results）のHTMLから、指標キー -> {年: (値, 予想かどうか)} を返す。"""
    soup = BeautifulSoup(html, "html.parser")
    out = {}
    for metric, dl in find_metric_dls(soup).items():
        series = parse_metric_dl(metric, dl)
        if series:
            out[metric] = series
    return out

In [ ]:
def format_value(metric: str, value: float) -> str:
    """金額（円）は整数、EPS・%系は小数2桁までの文字列にする（末尾の0は落とす）。"""
    if METRIC_UNITS[metric] == "yen" and metric != "eps":
        return str(int(round(value)))
    return f"{value:.2f}".rstrip("0").rstrip(".")


def load_existing(financials_csv: Path) -> dict[tuple[str, str, int], dict]:
    """既存のfinancials.csvを(code, metric, year) -> 行辞書のマップとして読み込む（無ければ空）。"""
    if not financials_csv.exists():
        return {}
    df = pd.read_csv(financials_csv, dtype=str).fillna("")
    # 全銘柄分（約50万行）でもすぐ読めるよう、iterrowsではなくto_dict("records")で回す
    return {(r["code"], r["metric"], int(r["year"])): r for r in df.to_dict("records")}


def save_financials_csv(rows: dict[tuple[str, str, int], dict], financials_csv: Path) -> None:
    financials_csv.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(list(rows.values()), columns=FINANCIAL_COLUMNS)
    df["_year"] = df["year"].astype(int)
    df = df.sort_values(["code", "metric", "_year"]).drop(columns=["_year"])
    df.to_csv(financials_csv, index=False)


def upsert_row(rows: dict, code: str, metric: str, year: int, value: float, is_forecast: bool, now: str) -> None:
    """(code, metric, year)の行を追加・更新する。既存が実績で新しい値が予想の場合は退行させない（C04と同じ規則）。"""
    key = (code, metric, year)
    existing = rows.get(key)
    if existing is not None and existing.get("is_forecast") == "False" and is_forecast:
        return
    rows[key] = {
        "code": code, "metric": metric, "year": str(year),
        "value": format_value(metric, value), "is_forecast": str(is_forecast), "updated_at": now,
    }

## 対象銘柄の読み込み

`stock/irbank.csv`から`status=ok`かつURLがある銘柄を対象にする。`FORCE_REFRESH_ALL=False`の場合、既に`stock/financials.csv`に1件でも行がある銘柄は除外する（中断後の再実行で続きから再開できる）。

In [ ]:
irbank_df = pd.read_csv(IRBANK_CSV, dtype=str).fillna("")
target_df = irbank_df[(irbank_df["status"] == "ok") & (irbank_df["url"] != "")]
all_target_codes = target_df.set_index("code")["url"].to_dict()
code_to_name = target_df.set_index("code")["name"].to_dict()

existing = load_existing(FINANCIALS_CSV)
fetched_codes = {code for (code, _metric, _year) in existing.keys()}

if TARGET_CODES:
    wanted = [str(c) for c in TARGET_CODES]
    missing = [c for c in wanted if c not in all_target_codes]
    if missing:
        print(f"irbank.csvにURLが無いため対象外: {', '.join(missing)}")
    target_codes = {c: all_target_codes[c] for c in wanted if c in all_target_codes}
else:
    pool = dict(all_target_codes)
    if HIGH_DIV_ONLY:
        labels_df = pd.read_csv(LABELS_CSV, dtype=str).fillna("")
        high_div_codes = set(labels_df.loc[labels_df["L_高配当"] == "1", "code"])
        pool = {c: u for c, u in pool.items() if c in high_div_codes}
    if FORCE_REFRESH_ALL:
        target_codes = pool
    else:
        target_codes = {c: u for c, u in pool.items() if c not in fetched_codes}

if TEST_LIMIT is not None:
    target_codes = dict(list(target_codes.items())[:TEST_LIMIT])

print(
    f"対象（IRBANK取得済み銘柄）: {len(all_target_codes)}銘柄 / 処理対象: {len(target_codes)}銘柄"
    f"（FORCE_REFRESH_ALL={FORCE_REFRESH_ALL}, HIGH_DIV_ONLY={HIGH_DIV_ONLY}, TARGET_CODES={TARGET_CODES}）"
)

## 取得実行

1銘柄の失敗（例外含む）は記録するだけで処理を継続する。`SAVE_EVERY`件ごとに`stock/financials.csv`へ逐次保存するので、途中でカーネルを止めてもそこまでの結果は残る（再実行すれば続きから再開する）。各銘柄の行に、取得できた指標を`指標名:年数`で表示する（取れなかった指標は表示されない。銀行・保険は`op_margin`が無いのが正常）。

In [ ]:
session = make_session()
succeeded = 0
no_data = []
failed = []

codes_list = list(target_codes.items())
for i, (code, url) in enumerate(codes_list):
    now = datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S")
    try:
        html = fetch_html(url, session)
        metrics = parse_financials(html)
    except Exception as e:
        print(f"[エラー] コード: {code}（{e}）")
        failed.append(code)
        if i < len(codes_list) - 1:
            polite_sleep(SLEEP_SEC)
        continue

    if not metrics:
        print(f"[情報なし] コード: {code}")
        no_data.append(code)
    else:
        for metric, series in metrics.items():
            for year, (value, is_forecast) in series.items():
                upsert_row(existing, code, metric, year, value, is_forecast, now)
        succeeded += 1
        summary = " ".join(f"{m}:{len(s)}" for m, s in metrics.items())
        print(f"[取得] {code} {code_to_name.get(code, '')} / {summary}")

    if i < len(codes_list) - 1:
        polite_sleep(SLEEP_SEC)

    if (i + 1) % SAVE_EVERY == 0:
        save_financials_csv(existing, FINANCIALS_CSV)
        print(f"--- 途中経過を保存しました（{i + 1}/{len(codes_list)}件処理） ---")

save_financials_csv(existing, FINANCIALS_CSV)
print(f"\n完了: 取得 {succeeded}件 / 情報なし {len(no_data)}件 / 失敗 {len(failed)}件（対象 {len(codes_list)}件中）")
if no_data:
    print(f"情報なしコード: {', '.join(no_data)}")
if failed:
    print(f"失敗コード: {', '.join(failed)}")

## 保存後の確認

銘柄コードを指定すると、指標×年の表（横持ち。末尾に`予`が付いているのは予想値）で確認できる。金額系（`revenue`・`op_cf`・`cash`）は見やすさのため億円単位で表示する（保存値は円単位）。確認できたら、データリポジトリ（`app_data`）の変更を自分でコミット・pushする。

In [ ]:
CHECK_CODE = "9432"

df = pd.read_csv(FINANCIALS_CSV, dtype=str).fillna("")
df = df[df["code"] == CHECK_CODE].copy()
if df.empty:
    print(f"{CHECK_CODE} のデータはありません")
else:
    def _display_value(row):
        v = float(row["value"])
        if row["metric"] in ("revenue", "op_cf", "cash"):
            v = round(v / 1e8, 1)  # 億円
        return f"{v:g}{' 予' if row['is_forecast'] == 'True' else ''}"

    df["display"] = df.apply(_display_value, axis=1)
    view = df.pivot(index="metric", columns="year", values="display").fillna("")
    view = view.reindex([m for m in METRIC_HEADINGS if m in view.index])
    view = view[sorted(view.columns, key=int)]
    print(f"{CHECK_CODE} {code_to_name.get(CHECK_CODE, '')}（revenue/op_cf/cashは億円）")
    display(view)

## アプリ用の軽量版を出力（`stock/financials_recent.csv`）

`stock/financials.csv`（全期間・縦持ち。全銘柄で約28MB）はアプリでそのまま読むには重いため、アプリの「データ」タブ＞「業績」モード（業績スコアの計算）用に、`RECENT_FROM_YEAR`年以降だけを1銘柄×1指標＝1行の横持ちにまとめた軽量版を出力する。IRBANKへのアクセスはしない（`financials.csv`から作り直すだけ）ので、取得の途中・後のどちらで実行してもよい。

- 列：`code, metric, {年}...`（年の列は`RECENT_FROM_YEAR`から最新年まで。値が無い年は空欄）。
- 予想値も実績と区別せずに入れる（業績スコアは予想値も計算に使う方針のため）。
- 金額系（`revenue`・`op_cf`・`cash`）は百万円単位の整数に丸める（スコアは傾向しか見ないため、桁を落としてファイルを小さくする）。
- `RECENT_FROM_YEAR`は、異常時耐性（コロナ）の基準年2019年と、直近10年の傾向の両方が入るように設定する。

In [ ]:
RECENT_FROM_YEAR = 2014
RECENT_CSV = DATA_DIR / "financials_recent.csv"
YEN_METRICS = {"revenue", "op_cf", "cash"}

full = pd.read_csv(FINANCIALS_CSV, dtype=str).fillna("")
full["year"] = full["year"].astype(int)
full = full[full["year"] >= RECENT_FROM_YEAR].copy()

def _compact_value(row):
    v = float(row["value"])
    if row["metric"] in YEN_METRICS:
        return str(int(round(v / 1e6)))
    return f"{v:.2f}".rstrip("0").rstrip(".")

full["compact"] = full.apply(_compact_value, axis=1)
wide = full.pivot(index=["code", "metric"], columns="year", values="compact").fillna("")
wide = wide[sorted(wide.columns)]
wide.columns = [str(c) for c in wide.columns]
wide = wide.reset_index()
wide.to_csv(RECENT_CSV, index=False)
print(f"保存しました: {RECENT_CSV}（{wide['code'].nunique()}銘柄 / {len(wide)}行 / {RECENT_CSV.stat().st_size / 1e6:.1f}MB）")